# `nn.Conv2d` from scratch

A 2D convolution slides a small learnable **kernel** over the spatial dimensions of an image and computes, at each position, a dot product between the kernel and the local patch it covers. It is the core building block of CNNs : local, translation-equivariant, and weight-shared (the same kernel is reused at every position).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## 1. The operation

Given an input $x \in \mathbb{R}^{B \times C_{in} \times H \times W}$, a weight $W \in \mathbb{R}^{C_{out} \times C_{in} \times k_H \times k_W}$ and a bias $b \in \mathbb{R}^{C_{out}}$, the output $y \in \mathbb{R}^{B \times C_{out} \times H_{out} \times W_{out}}$ is :

$$y[b, c_o, i, j] = b[c_o] + \sum_{c_i=1}^{C_{in}} \sum_{u=1}^{k_H} \sum_{v=1}^{k_W} x[b,\ c_i,\ i \cdot s + u,\ j \cdot s + v] \; W[c_o,\ c_i,\ u,\ v]$$

where $s$ is the **stride**. Each output channel $c_o$ has its own kernel of shape $(C_{in}, k_H, k_W)$ ; the kernel is **the same** at every spatial position $(i, j)$ — this is **weight sharing**, the reason a CNN has few parameters.

## 2. Output spatial size

With padding $p$ and stride $s$ :

$$H_{out} = \left\lfloor \frac{H + 2p - k_H}{s} \right\rfloor + 1, \qquad W_{out} = \left\lfloor \frac{W + 2p - k_W}{s} \right\rfloor + 1$$

Special cases :
- `kernel=k, stride=1, padding=k//2` (odd $k$) → output **same** size as input
- `kernel=stride=k, padding=0` → each output position covers **one non-overlapping $k\times k$ patch** (this is the "patchify" conv used in ViT/CPC)

## 3. From-scratch implementation (im2col)

The trick : extract every sliding patch into a column (`unfold` = im2col), flatten the kernel into a matrix, and the convolution becomes a single **matrix multiplication**.

- `unfold` turns $x$ into patches of shape $(B,\ C_{in} \cdot k_H \cdot k_W,\ L)$ where $L = H_{out} \cdot W_{out}$
- the weight becomes $(C_{out},\ C_{in} \cdot k_H \cdot k_W)$
- multiply : $(C_{out}, K) \times (B, K, L) \to (B, C_{out}, L)$, then reshape to $(B, C_{out}, H_{out}, W_{out})$

In [ ]:
def conv2d_scratch(x, weight, bias, stride=1, padding=0):
    # x      : (B, C_in, H, W)
    # weight : (C_out, C_in, kH, kW)
    # bias   : (C_out,)
    B, C_in, H, W = x.shape
    C_out, _, kH, kW = weight.shape

    H_out = (H + 2 * padding - kH) // stride + 1
    W_out = (W + 2 * padding - kW) // stride + 1

    # im2col : every sliding patch becomes a column
    patches = F.unfold(x, kernel_size=(kH, kW), stride=stride, padding=padding)  # (B, C_in*kH*kW, L)

    # flatten the kernel into a matrix
    W_mat = weight.view(C_out, -1)                                    # (C_out, C_in*kH*kW)

    # convolution = matrix multiplication
    out = torch.einsum('ok,bkl->bol', W_mat, patches)                 # (B, C_out, L)
    out = out + bias.view(1, -1, 1)
    out = out.view(B, C_out, H_out, W_out)
    return out

## 4. Verify against `nn.Conv2d`

In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, stride=2, padding=1)

x = torch.randn(4, 3, 16, 16)
y_torch   = conv(x)
y_scratch = conv2d_scratch(x, conv.weight, conv.bias, stride=2, padding=1)

print("torch   :", tuple(y_torch.shape))
print("scratch :", tuple(y_scratch.shape))
print("max abs diff :", (y_torch - y_scratch).abs().max().item())
print("match :", torch.allclose(y_torch, y_scratch, atol=1e-5))

## 5. Parameters

A `nn.Conv2d(C_in, C_out, k)` has :

$$\underbrace{C_{out} \cdot C_{in} \cdot k_H \cdot k_W}_{\text{weight}} \;+\; \underbrace{C_{out}}_{\text{bias}}$$

Crucially, this **does not depend on the image size** $H, W$ — the same kernel is reused at every position. That is why CNNs are parameter-efficient compared to fully-connected layers (an MLP on a flattened image scales with $H \cdot W$).

In [ ]:
conv = nn.Conv2d(3, 8, kernel_size=3)
params    = sum(p.numel() for p in conv.parameters())
params_th = 8 * 3 * 3 * 3 + 8       # C_out*C_in*kH*kW + C_out
print(f"empirical : {params}")
print(f"theory    : {params_th}")
print(f"match     : {params == params_th}")

## 6. The three knobs

| Knob | Effect |
|---|---|
| **kernel_size** $k$ | size of the receptive field at this layer ; bigger $k$ = more context, more params ($\propto k^2$) |
| **stride** $s$ | downsampling factor ; $s=2$ halves $H, W$ |
| **padding** $p$ | adds zeros around the border ; $p = k//2$ keeps the size unchanged |

**1×1 convolution** ($k=1$) : no spatial mixing at all — it is just a per-position linear layer across channels ($C_{in} \to C_{out}$). Used to change the channel dimension cheaply (e.g. in the CPC per-patch encoder).

**Receptive field** : stacking convolutions grows the receptive field. Two $3\times3$ convs see a $5\times5$ region ; three see $7\times7$. Depth buys global context without large kernels.